# Aula 6 — Alocação: do Sinal aos Pesos

**Intensivão Quant AI — ImpactUFSCar**

---

Até agora sabemos **quem** comprar (o sinal nos diz). Hoje aprendemos **quanto** alocar em cada ativo — e por que essa decisão importa tanto quanto o sinal em si.

Ao final desta aula você terá:
- Entendido o portfólio como álgebra linear: `wᵀr` e `wᵀΣw`
- Implementado signal-weighted e inverse volatility
- Comparado três esquemas de alocação com `calcular_metricas()`

In [ ]:
# ── CONFIGURAÇÃO DO AMBIENTE ─────────────────────────────────────────────────
# Este notebook roda no VS Code (Jupyter local) E no Google Colab.
# Execute esta célula primeiro — ela detecta o ambiente e configura os caminhos.

import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # ── Google Colab ──────────────────────────────────────────────────────────
    print("Ambiente detectado: Google Colab")

    # Montar Google Drive para persistir os dados entre sessões
    from google.colab import drive
    drive.mount('/content/drive')

    # Instalar dependências não incluídas no Colab por padrão
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'yfinance', 'pyarrow'], check=False)

    # Clonar o repositório do curso (pula automaticamente se já existir)
    REPO_DIR = '/content/intensivao-quant-ai'
    if not os.path.exists(REPO_DIR):
        print("Clonando repositório do curso...")
        subprocess.run([
            'git', 'clone',
            'https://github.com/fmaldonadoo/intensivao-quant-ai.git',
            REPO_DIR
        ])
        print("Repositório clonado com sucesso.")

    # Pasta de dados no Google Drive — persiste entre sessões do Colab
    DADOS_DIR = '/content/drive/MyDrive/intensivao_quant/dados'
    os.makedirs(DADOS_DIR, exist_ok=True)
    print(f"Dados serão salvos em: {DADOS_DIR}")

else:
    # ── VS Code / Jupyter local ───────────────────────────────────────────────
    print("Ambiente detectado: VS Code / Jupyter local")

    # Sobe a árvore de diretórios até encontrar a raiz do repositório (.git)
    _p = os.path.abspath(os.getcwd())
    while _p != os.path.dirname(_p):
        if os.path.exists(os.path.join(_p, '.git')):
            break
        _p = os.path.dirname(_p)

    DADOS_DIR = os.path.join(_p, 'dados')
    os.makedirs(DADOS_DIR, exist_ok=True)
    print(f"Dados serão salvos em: {DADOS_DIR}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Dados das aulas anteriores
ret_diarios  = pd.read_parquet(os.path.join(DADOS_DIR, 'retornos_diarios_limpo.parquet'))
ret_mensais  = pd.read_parquet(os.path.join(DADOS_DIR, 'retornos_mensais_limpo.parquet'))
sinal        = pd.read_parquet(os.path.join(DADOS_DIR, 'sinal_v1.parquet'))
ranking      = sinal.rank(axis=1, ascending=True, pct=True)

# Benchmark
ibov_raw = yf.download('^BVSP', start='2010-01-01', end='2024-12-31',
                        auto_adjust=True, progress=False)['Close'].squeeze()
ret_ibov = np.log(ibov_raw / ibov_raw.shift(1)).resample('ME').sum()

# Funções da Aula 5
from scipy.stats import linregress

def calcular_metricas(retornos, benchmark=None, nome='Estratégia', rf_mensal=0.0):
    n_anos    = len(retornos) / 12
    ret_total = np.exp(retornos.sum()) - 1
    cagr      = (1 + ret_total) ** (1 / n_anos) - 1
    vol_anual = retornos.std() * np.sqrt(12)
    excesso   = retornos - rf_mensal
    sharpe    = (excesso.mean() / excesso.std()) * np.sqrt(12)
    ret_neg   = excesso[excesso < 0]
    sortino   = (excesso.mean() * 12) / (ret_neg.std() * np.sqrt(12)) if len(ret_neg) > 1 else np.nan
    valor     = np.exp(retornos.cumsum())
    mdd       = ((valor - valor.cummax()) / valor.cummax()).min()
    calmar    = cagr / abs(mdd) if mdd != 0 else np.nan
    alpha, beta_val = np.nan, np.nan
    if benchmark is not None:
        idx = retornos.index.intersection(benchmark.index)
        if len(idx) > 10:
            sl, ic, *_ = linregress(benchmark[idx], retornos[idx])
            beta_val, alpha = sl, ic * 12
    return dict(nome=nome, cagr=cagr, vol_anual=vol_anual, sharpe=sharpe,
                sortino=sortino, max_drawdown=mdd, calmar=calmar,
                alpha_anual=alpha, beta=beta_val)

def exibir_metricas(*ms):
    fmt = dict(cagr='{:.1%}', vol_anual='{:.1%}', sharpe='{:.2f}',
               sortino='{:.2f}', max_drawdown='{:.1%}', calmar='{:.2f}',
               alpha_anual='{:.2%}', beta='{:.2f}')
    labels = dict(cagr='CAGR', vol_anual='Vol Anual', sharpe='Sharpe',
                  sortino='Sortino', max_drawdown='Max Drawdown',
                  calmar='Calmar', alpha_anual='Alpha', beta='Beta')
    nomes = [d['nome'] for d in ms]
    w = max(14, max(len(n) for n in nomes) + 2)
    h = f"{'Métrica':<22}" + ''.join(f"{n:>{w}}" for n in nomes)
    print(h); print('─' * len(h))
    for k, lb in labels.items():
        linha = f"{lb:<22}"
        for d in ms:
            v = d.get(k, np.nan)
            s = fmt[k].format(v) if not (isinstance(v, float) and np.isnan(v)) else 'N/A'
            linha += f"{s:>{w}}"
        print(linha)

N_ATIVOS = 10
print("Setup concluído.")

## 1. O portfólio como álgebra linear

Um portfólio é um vetor de pesos $\mathbf{w} = [w_1, w_2, \ldots, w_N]^\top$ onde $\sum_i w_i = 1$.

### Retorno do portfólio

O retorno do portfólio é o **produto interno** entre o vetor de pesos e o vetor de retornos:

$$r_p = \mathbf{w}^\top \mathbf{r} = \sum_{i=1}^{N} w_i \cdot r_i$$

No `numpy`, isso é `np.dot(w, r)` ou `w @ r`.

### Variância do portfólio

O risco do portfólio não é a soma dos riscos individuais — depende de como os ativos se movem juntos. Isso é capturado pela **matriz de covariância** $\Sigma$:

$$\Sigma_{ij} = \text{Cov}(r_i, r_j)$$

- Diagonal de $\Sigma$: variâncias individuais ($\sigma_i^2$)
- Fora da diagonal: covariâncias entre pares de ativos

A variância do portfólio é a **forma quadrática**:

$$\sigma_p^2 = \mathbf{w}^\top \Sigma \mathbf{w}$$

**Por que isso importa:** dois portfólios com os mesmos ativos mas pesos diferentes podem ter riscos totais muito distintos — dependendo das correlações entre os ativos.

In [ ]:
# Demonstração: wᵀΣw para carteiras diferentes
# Usamos um mês específico para ilustrar

data_exemplo = '2022-12-31'

# Ações com sinal válido nessa data
sinal_mes = sinal.loc[data_exemplo].dropna()
top_10    = sinal_mes.nlargest(10).index.tolist()

# Retornos diários dos últimos 252 dias para calcular Σ
janela_cov = ret_diarios.loc[:data_exemplo].iloc[-252:][top_10].dropna(axis=1)
ativos_cov = janela_cov.columns.tolist()
n = len(ativos_cov)

Sigma = janela_cov.cov().values  # matriz N×N

# Três vetores de pesos para comparar
w_equal = np.ones(n) / n
w_conc  = np.zeros(n); w_conc[0] = 1.0          # tudo em um ativo
w_rand  = np.abs(np.random.randn(n)); w_rand /= w_rand.sum()  # aleatório

def var_portfolio(w, Sigma):
    return w @ Sigma @ w

print(f"Matriz Σ: {Sigma.shape[0]}×{Sigma.shape[1]}")
print()
print("Variância do portfólio (wᵀΣw) — anualizada:")
print(f"  Equal weight:   {var_portfolio(w_equal, Sigma) * 252:.4f}  "
      f"→ vol = {np.sqrt(var_portfolio(w_equal, Sigma) * 252):.2%}")
print(f"  Concentrado:    {var_portfolio(w_conc,  Sigma) * 252:.4f}  "
      f"→ vol = {np.sqrt(var_portfolio(w_conc,  Sigma) * 252):.2%}")
print(f"  Aleatório:      {var_portfolio(w_rand,  Sigma) * 252:.4f}  "
      f"→ vol = {np.sqrt(var_portfolio(w_rand,  Sigma) * 252):.2%}")

## 2. Markowitz — a intuição sem a matemática pesada

Harry Markowitz (Prêmio Nobel 1990) formalizou matematicamente algo intuitivo: **diversificação reduz risco quando os ativos não se movem perfeitamente juntos**.

Para dois ativos com correlação $\rho$:

$$\sigma_p^2 = w_1^2 \sigma_1^2 + w_2^2 \sigma_2^2 + 2 w_1 w_2 \rho \sigma_1 \sigma_2$$

- Se $\rho = 1$: sem benefício de diversificação
- Se $\rho = 0$: diversificação reduz risco
- Se $\rho = -1$: diversificação máxima (risco pode ir a zero)

O conjunto de portfólios com máximo retorno para cada nível de risco forma a **fronteira eficiente**.

In [ ]:
# Visualização 1: efeito da correlação na volatilidade do portfólio
w = 0.5  # 50% em cada ativo
s1, s2 = 0.30, 0.25  # volatilidades anualizadas

correlacoes = np.linspace(-1, 1, 200)
vol_port = np.sqrt(w**2 * s1**2 + w**2 * s2**2 + 2 * w * w * correlacoes * s1 * s2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(correlacoes, vol_port, color='steelblue', linewidth=2)
axes[0].axhline(w * s1 + w * s2, color='gray', linestyle='--',
                label=f'sem diversif. = {w*s1+w*s2:.1%}')
axes[0].set_xlabel('Correlação entre os ativos')
axes[0].set_ylabel('Volatilidade do portfólio')
axes[0].set_title('Diversificação: benefício cresce com menor correlação')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
axes[0].legend()

# Visualização 2: fronteira eficiente via Monte Carlo (5 ativos)
cinco_ativos = [a for a in ativos_cov[:5]]
mu  = ret_diarios[cinco_ativos].mean().values * 252
Sig = ret_diarios[cinco_ativos].cov().values  * 252

np.random.seed(42)
N_SIMS = 5000
rets_sim, vols_sim, sharpes_sim = [], [], []

for _ in range(N_SIMS):
    w_sim = np.abs(np.random.randn(5))
    w_sim /= w_sim.sum()
    r_sim = w_sim @ mu
    v_sim = np.sqrt(w_sim @ Sig @ w_sim)
    rets_sim.append(r_sim)
    vols_sim.append(v_sim)
    sharpes_sim.append(r_sim / v_sim)

sc = axes[1].scatter(vols_sim, rets_sim, c=sharpes_sim,
                      cmap='RdYlGn', alpha=0.4, s=5)
plt.colorbar(sc, ax=axes[1], label='Sharpe')
axes[1].set_xlabel('Volatilidade anual')
axes[1].set_ylabel('Retorno anual esperado')
axes[1].set_title('Fronteira eficiente — 5.000 portfólios aleatórios')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

plt.tight_layout()
plt.show()

## 3. Por que equal weight frequentemente vence a otimização

Se Markowitz é matematicamente ótimo, por que não o usamos?

DeMiguel, Garlappi & Uppal (2009) — *"Optimal Versus Naive Diversification"* — testaram 14 modelos de otimização contra o portfólio $1/N$ (equal weight) em dados reais. Resultado: **nenhum modelo bateu o equal weight de forma consistente**.

**Por quê:**

1. **Erro de estimação da média:** estimar o retorno esperado de cada ativo com precisão suficiente requer centenas de anos de dados — impossível na prática
2. **Erro de estimação da covariância:** a matriz $\Sigma$ tem $N(N+1)/2$ parâmetros. Para 50 ativos, são 1.275 parâmetros estimados com ruído
3. **Otimização amplifica erros:** os pesos da otimização são funções dos parâmetros estimados — quando os parâmetros têm erro, os pesos resultantes são muito instáveis

**Resultado prático:** $1/N$ é um benchmark difícil de bater. Vamos usá-lo como baseline e testar se esquemas mais sofisticados melhoram.

## 4. Signal-weighted

Em vez de dar peso igual a todos os selecionados, damos **peso proporcional ao rank** do sinal. Ativos com sinal mais forte recebem mais capital.

Para os top $N$ ativos com ranks $\tilde{r}_1 \geq \tilde{r}_2 \geq \ldots \geq \tilde{r}_N$:

$$w_i = \frac{\tilde{r}_i}{\sum_{j=1}^{N} \tilde{r}_j}$$

Isso concentra mais capital nos ativos com sinal mais forte, mantendo a soma dos pesos = 1.

In [ ]:
def calcular_pesos_signal_weighted(ranking_row, N=10):
    validos = ranking_row.dropna()
    if len(validos) < N:
        return pd.Series(0.0, index=ranking_row.index)

    top_n  = validos.nlargest(N)
    pesos  = pd.Series(0.0, index=ranking_row.index)
    pesos[top_n.index] = top_n / top_n.sum()  # normalizado pelo total
    return pesos


pesos_sw = ranking.apply(calcular_pesos_signal_weighted, axis=1, N=N_ATIVOS)

# Verificação
print("Pesos signal-weighted — último mês (top 5):")
ult = pesos_sw.iloc[-1]
print(ult[ult > 0].sort_values(ascending=False).head()
      .rename(lambda x: x.replace('.SA','')).map('{:.1%}'.format))
print(f"Soma: {ult.sum():.4f}")

In [ ]:
# Comparação visual: distribuição dos pesos em um mês
from pandas import read_parquet
pesos_ew = read_parquet(os.path.join(DADOS_DIR, 'pesos_v1.parquet'))  # equal weight da Aula 4

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (pesos_plot, titulo) in zip(axes, [
    (pesos_ew,  'Equal Weight'),
    (pesos_sw,  'Signal Weighted')
]):
    ult = pesos_plot.iloc[-1]
    ult_nz = ult[ult > 0].sort_values(ascending=False)
    ult_nz.rename(lambda x: x.replace('.SA', '')).plot(
        kind='bar', ax=ax, color='steelblue', width=0.7
    )
    ax.set_title(f'{titulo} — {pesos_plot.index[-1].strftime("%b/%Y")}')
    ax.set_ylabel('Peso')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.axhline(1/N_ATIVOS, color='red', linestyle='--',
               linewidth=1, label=f'equal = {1/N_ATIVOS:.0%}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 5. Inverse Volatility — aproximação de risk parity

**Risk parity** é o princípio de que cada ativo deve contribuir igualmente para o risco total do portfólio.

A contribuição de risco do ativo $i$ é:

$$\text{RC}_i = w_i \cdot \frac{(\Sigma \mathbf{w})_i}{\mathbf{w}^\top \Sigma \mathbf{w}}$$

Risk parity exige $\text{RC}_i = 1/N$ para todo $i$ — o que requer resolver um sistema não-linear.

**Aproximação prática — Inverse Volatility:**

Se ignorarmos as correlações ($\Sigma \approx$ diagonal), a solução simplifica para:

$$w_i \propto \frac{1}{\sigma_i}$$

Ativos mais voláteis recebem pesos menores; ativos mais estáveis, pesos maiores. É rápido, intuitivo e funciona surpreendentemente bem na prática.

In [ ]:
def calcular_pesos_inv_vol(ranking_row, ret_diarios, data, N=10, janela=63):
    """
    Top N por sinal, pesos inversamente proporcionais à volatilidade rolling.
    Janela padrão: 63 dias (~3 meses).
    """
    validos = ranking_row.dropna()
    if len(validos) < N:
        return pd.Series(0.0, index=ranking_row.index)

    top_n_tickers = validos.nlargest(N).index.tolist()

    # Volatilidade rolling até a data de rebalanceamento
    hist = ret_diarios.loc[:data, top_n_tickers].iloc[-janela:]
    vol  = hist.std()
    vol  = vol.replace(0, np.nan).dropna()

    inv_vol = 1 / vol
    pesos   = pd.Series(0.0, index=ranking_row.index)
    pesos[inv_vol.index] = inv_vol / inv_vol.sum()
    return pesos


print("Calculando pesos inverse volatility (pode levar alguns segundos)...")
pesos_iv_list = []
for data in ranking.index:
    row = ranking.loc[data]
    p   = calcular_pesos_inv_vol(row, ret_diarios, data, N=N_ATIVOS)
    pesos_iv_list.append(p)

pesos_iv = pd.DataFrame(pesos_iv_list, index=ranking.index)
print("Concluído.")
print(f"Soma dos pesos (verificação): {pesos_iv.sum(axis=1)[pesos_iv.sum(axis=1) > 0].mean():.4f}")

## 6. Variância realizada — wᵀΣw para cada esquema

Vamos calcular a variância ex-ante (prevista pelo modelo) e comparar com a volatilidade ex-post (realizada). Isso mostra quanto cada esquema de alocação realmente controla o risco.

In [ ]:
def vol_exante_mensal(pesos_df, ret_diarios, janela=63):
    """Calcula a vol ex-ante (wᵀΣw) mês a mês."""
    vols = []
    for data in pesos_df.index:
        w = pesos_df.loc[data]
        ativos = w[w > 0].index.tolist()
        if len(ativos) < 2:
            vols.append(np.nan)
            continue
        hist  = ret_diarios.loc[:data, ativos].iloc[-janela:].dropna(axis=1)
        at_ok = hist.columns.tolist()
        w_ok  = w[at_ok].values
        if w_ok.sum() == 0:
            vols.append(np.nan)
            continue
        w_ok /= w_ok.sum()
        Sig   = hist.cov().values
        var   = w_ok @ Sig @ w_ok
        vols.append(np.sqrt(var * 252))  # vol anualizada
    return pd.Series(vols, index=pesos_df.index)


print("Calculando vol ex-ante para os três esquemas...")
vol_ea_ew = vol_exante_mensal(pesos_ew, ret_diarios)
vol_ea_sw = vol_exante_mensal(pesos_sw, ret_diarios)
vol_ea_iv = vol_exante_mensal(pesos_iv, ret_diarios)
print("Concluído.")

fig, ax = plt.subplots(figsize=(13, 4))
vol_ea_ew.plot(ax=ax, label='Equal Weight',       alpha=0.8)
vol_ea_sw.plot(ax=ax, label='Signal Weighted',    alpha=0.8)
vol_ea_iv.plot(ax=ax, label='Inverse Vol',        alpha=0.8)
ax.set_title('Volatilidade ex-ante (wᵀΣw) — comparação dos três esquemas')
ax.set_ylabel('Volatilidade anualizada prevista')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nVol ex-ante média:")
print(f"  Equal Weight:   {vol_ea_ew.mean():.1%}")
print(f"  Signal Weighted: {vol_ea_sw.mean():.1%}")
print(f"  Inverse Vol:    {vol_ea_iv.mean():.1%}")

## 7. Comparação dos três esquemas

In [ ]:
def retorno_carteira(pesos_df, ret_mensais):
    ret = (pesos_df * ret_mensais).sum(axis=1)
    return ret[pesos_df.sum(axis=1) > 0]

ret_ew = retorno_carteira(pesos_ew, ret_mensais)
ret_sw = retorno_carteira(pesos_sw, ret_mensais)
ret_iv = retorno_carteira(pesos_iv, ret_mensais)

idx_comum = ret_ew.index.intersection(ret_sw.index).intersection(ret_iv.index)
ret_ew, ret_sw, ret_iv = ret_ew[idx_comum], ret_sw[idx_comum], ret_iv[idx_comum]
ret_bm = ret_ibov.reindex(idx_comum).dropna()
ret_ew, ret_sw, ret_iv = (
    ret_ew.reindex(ret_bm.index),
    ret_sw.reindex(ret_bm.index),
    ret_iv.reindex(ret_bm.index),
)

# Curvas acumuladas
fig, ax = plt.subplots(figsize=(13, 5))
np.exp(ret_ew.cumsum()).plot(ax=ax, label='Equal Weight',     linewidth=2)
np.exp(ret_sw.cumsum()).plot(ax=ax, label='Signal Weighted',  linewidth=2)
np.exp(ret_iv.cumsum()).plot(ax=ax, label='Inverse Vol',      linewidth=2)
np.exp(ret_bm.cumsum()).plot(ax=ax, label='IBOVESPA',
                              linewidth=1.5, linestyle='--', color='gray')
ax.set_title('Retorno acumulado — comparação de esquemas de alocação')
ax.set_ylabel('Valor (base 1.0)')
ax.legend()
plt.tight_layout()
plt.show()

# Métricas
print()
m_ew = calcular_metricas(ret_ew, ret_bm, 'Equal Weight')
m_sw = calcular_metricas(ret_sw, ret_bm, 'Signal Weighted')
m_iv = calcular_metricas(ret_iv, ret_bm, 'Inverse Vol')
m_bm = calcular_metricas(ret_bm, nome='IBOVESPA')
exibir_metricas(m_ew, m_sw, m_iv, m_bm)

## 8. Qual esquema escolher?

Não existe resposta universal. Cada esquema tem trade-offs:

| Esquema | Vantagem | Desvantagem |
|---|---|---|
| Equal weight | Simples, transparente, difícil de bater | Ignora risco e força do sinal |
| Signal weighted | Concentra mais em sinais fortes | Mais turnover, mais concentrado |
| Inverse volatility | Controla risco, defensivo | Pode subalocar em ativos com boa relação retorno/vol |

Para o relatório do desafio, o que importa é **você saber justificar a escolha** — não existe a escolha certa, existe a escolha fundamentada.

Salve o esquema que seu grupo escolheu para avançar.

In [ ]:
# Salve o esquema escolhido como 'pesos_v2'
# Troque 'pesos_sw' pelo esquema escolhido pelo seu grupo
ESQUEMA_ESCOLHIDO = pesos_sw   # ← altere aqui se quiser
NOME_ESQUEMA      = 'signal_weighted'

ESQUEMA_ESCOLHIDO.to_parquet(os.path.join(DADOS_DIR, 'pesos_v2.parquet'))
retorno_carteira(ESQUEMA_ESCOLHIDO, ret_mensais).to_frame('ret_momentum').to_parquet(
    os.path.join(DADOS_DIR, 'retorno_carteira_v2.parquet')
)

print(f"Esquema '{NOME_ESQUEMA}' salvo como pesos_v2.parquet")

## Resumo da aula

| Conceito | Implementação |
|---|---|
| Retorno do portfólio | `(pesos * ret_mensais).sum(axis=1)` = **wᵀr** |
| Variância do portfólio | `w @ Sigma @ w` = **wᵀΣw** |
| Fronteira eficiente | Conjunto de portfólios ótimos — visualizado via Monte Carlo |
| Por que equal weight | Erro de estimação torna otimização instável na prática |
| Signal weighted | Pesos ∝ rank — concentra em sinais mais fortes |
| Inverse volatility | Pesos ∝ 1/σ — aproximação de risk parity |

---

## Para replicar em casa

1. Rode o notebook e observe as métricas dos três esquemas
2. Implemente risk parity completo (contribuições iguais): `w_i ∝ 1/σ_i` já é a versão simplificada — pesquise `scipy.optimize.minimize` para a versão exata
3. Teste `N_ATIVOS = 15` com inverse vol — o Sharpe melhora ou piora?

---

*ImpactUFSCar — Diretoria de Quant*